In [ ]:
# 计算文本的表征，存成文件
from transformers import AutoTokenizer,AutoModel
import torch
import numpy as np
from tqdm import tqdm
from datasets import load_dataset,Dataset

# output_file = "draw-data/emb/C3-bb-emb.npz"

# model_list = [
#     {"model_name":"bert-base-uncased","title":"BERT"},
#     {"model_name":"princeton-nlp/unsup-simcse-bert-base-uncased","title":"SimCSE"},
#     {"model_name":"sosuke/ease-bert-base-uncased","title":"EASE"},
#     {"model_name":"ffgcc/esimcse-bert-base-uncased","title":"Ours"}
# ]

# output_file = "draw-data/emb/C3-bl-emb.npz"

# model_list = [
#     {"model_name":"bert-large-uncased","title":"BERT"},
#     {"model_name":"princeton-nlp/unsup-simcse-bert-large-uncased","title":"SimCSE"},
#     {"model_name":"../result/ease-bert-large/","title":"EASE"},
#     {"model_name":"ffgcc/esimcse-bert-large-uncased","title":"Ours"}
# ]

# output_file = "draw-data/emb/C3-rb-emb.npz"

# model_list = [
#     {"model_name":"FacebookAI/roberta-base","title":"RoBERTa"},
#     {"model_name":"princeton-nlp/unsup-simcse-roberta-base","title":"SimCSE"},
#     {"model_name":"sosuke/ease-roberta-base","title":"EASE"},
#     {"model_name":"ffgcc/esimcse-roberta-base","title":"Ours"}
# ]


output_file = "draw-data/emb/C3-rl-emb.npz"

model_list = [
    {"model_name":"FacebookAI/roberta-large","title":"RoBERTa"},
    {"model_name":"princeton-nlp/unsup-simcse-roberta-large","title":"SimCSE"},
    {"model_name":"../result/ease-roberta-large","title":"EASE"},
    {"model_name":"ffgcc/esimcse-roberta-large","title":"Ours"}
]


dataset_name = "mteb/stsbenchmark-sts"
dataset = load_dataset(dataset_name, split="train")
sent_list = dataset['sentence1'] + dataset['sentence2']
dataset = Dataset.from_dict({"text": sent_list})


bs = 512    # 以bs为单位进行推理
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

emb_dict = {}

for model_item in tqdm(model_list):

    tokenizer = AutoTokenizer.from_pretrained(model_item["model_name"])
    model = AutoModel.from_pretrained(model_item["model_name"]).to(device)
    emb = []

    def prepare(examples):
        return tokenizer(examples["text"], padding="longest", truncation=True, max_length=512)
    dataset = dataset.map(prepare, batched=True)

    for batch in tqdm(dataset.batch(bs)):
        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]

        # 对齐
        max_len = max([len(ids) for ids in input_ids])
        input_ids = [ids + [tokenizer.pad_token_id] * (max_len - len(ids)) for ids in input_ids]
        attention_mask = [mask + [0] * (max_len - len(mask)) for mask in attention_mask]

        input_ids = torch.tensor(input_ids, dtype=torch.long).to(device)
        attention_mask = torch.tensor(attention_mask, dtype=torch.long).to(device)

        with torch.no_grad():
            outputs = model(input_ids, attention_mask=attention_mask)

        last_hidden_state = outputs.last_hidden_state
        pooler_output =last_hidden_state[:,0,:]
        emb.append(pooler_output.cpu().numpy())
    emb_dict[model_item["title"]] = np.concatenate(emb, axis=0)

np.savez(output_file, **emb_dict)
print("Save to", output_file)

In [ ]:
# 跨语言文本表征向量计算
from transformers import AutoTokenizer,AutoModel
import torch
import numpy as np
from tqdm import tqdm
from datasets import load_dataset,Dataset

model_name = "/root/autodl-tmp/e114_20"
output_file = f"draw-data/e114_20-multi-stsb-train.npz"

# model_name = "sentence-transformers/LaBSE"
# output_file = f"draw-data/emb/labse-multi-stsb-train.npz"

dataset_name = "mteb/stsb_multi_mt"
# lang_list = ["en","de","es","fr","it","nl","pl","pt","ru","zh"]
lang_list = ["en","de","fr","ru","zh"]

sample_num = 256

lang_sent_dict = {}
for lang in lang_list:
    dataset = load_dataset(dataset_name, name=lang, split="train",trust_remote_code=True)
    # 每个都只要sentence1，并采样1000个
    # 相同种子采样完对应id仍然是并行语料
    dataset = dataset.shuffle(seed=918).select(range(sample_num))
    lang_sent_dict[lang] = dataset['sentence1']

dataset = Dataset.from_dict(lang_sent_dict)

max_seq_length = 32
bs = 256    # 以bs为单位进行推理
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)

# 用于存储所有文本的表征
lang_emb_dict = {}

def prepare(examples):
    return tokenizer(examples["text"], padding=False, truncation=True, max_length=max_seq_length)

for lang in tqdm(lang_list):
    lang_dataset = Dataset.from_dict({"text": lang_sent_dict[lang]})
    lang_dataset = lang_dataset.map(prepare, batched=True)
    lang_all_embs = []

    for batch in tqdm(lang_dataset.batch(bs)):
        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]

        # 对齐
        max_len = max([len(ids) for ids in input_ids])
        input_ids = [ids + [tokenizer.pad_token_id] * (max_len - len(ids)) for ids in input_ids]
        attention_mask = [mask + [0] * (max_len - len(mask)) for mask in attention_mask]

        input_ids = torch.tensor(input_ids, dtype=torch.long).to(device)
        attention_mask = torch.tensor(attention_mask, dtype=torch.long).to(device)

        with torch.no_grad():
            outputs = model(input_ids, attention_mask=attention_mask)

        last_hidden_state = outputs.last_hidden_state
        pooler_output =last_hidden_state[:,0,:]
        lang_all_embs.append(pooler_output.cpu().numpy())
    lang_emb_dict[lang] = np.concatenate(lang_all_embs, axis=0)


np.savez(output_file, **lang_emb_dict)
print(f"Saved to {output_file}")

In [ ]:
import spacy
from transformers import AutoTokenizer
import numpy as np
from tqdm import tqdm

data_dir = 'draw-data/'

nlp = spacy.load('en_core_web_sm')
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

# 计算原句子
input_file = '../data/wiki1m_for_simcse.txt'
with open(input_file, 'r', encoding='utf-8') as f:
    sent_list = f.read().splitlines()

# 将数据拆分成较小的批次
batch_size = 5000
docs = list(tqdm(nlp.pipe(sent_list, batch_size=batch_size, disable=["ner", "parser", "textcat"], n_process=10), total=len(sent_list),desc='计算句子长度'))
sent_l_list = [len(doc) for doc in docs]

sent_l_arr = np.array(sent_l_list)
np.save(data_dir + 'c4-句子长度数组.npy', sent_l_arr)

token_l_list = []

for sent in tqdm(sent_list, desc='计算句子token长度'):
    token_l_list.append(len(tokenizer.tokenize(sent)))

token_l_arr = np.array(token_l_list)
np.save(data_dir + 'c4-句子token长度数组.npy', token_l_arr)

In [ ]:
# title和abstract转储
# 计算title和abstract的长度
import os
import sys
import json
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
import numpy as np
from knowledge.backend import MySQLClient
from tqdm import tqdm
import spacy
from transformers import AutoTokenizer

data_dir = 'draw-data/'
if os.path.exists(data_dir) == False:
    os.makedirs(data_dir)

nlp = spacy.load('en_core_web_sm')
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

MySQL = MySQLClient()

title_list = []
abstrct_list = []
offset = 0
limit = 1000
print('开始加载数据')
while True:
    sent_list = MySQL.batch_get_page_info_title_abstract(offset,limit)
    offset+=limit
    if not sent_list:
        break
    for sent in sent_list:
        # title_list.append(sent[1])
        abstrct_list.append(sent[2])

    # 每10万打印一次
    if offset % 100000 == 0:
        print(f'已加载{offset}条数据')
# # 存下来
# with open(data_dir + 'c4-title.txt', 'w', encoding='utf-8') as f:
#     for sent in title_list:
#         if sent:
#             f.write(sent + '\n')
with open(data_dir + 'c4-abstract.json', 'w', encoding='utf-8') as f:
    json.dump(abstrct_list, f)
print('加载数据完成')

In [ ]:
# 计算title和abstract的长度
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
import numpy as np
from knowledge.backend import MySQLClient
from tqdm import tqdm
import spacy
from transformers import AutoTokenizer
import json

data_dir = 'draw-data/'
if os.path.exists(data_dir) == False:
    os.makedirs(data_dir)

nlp = spacy.load('en_core_web_sm')
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

print('开始加载数据')
# with open(data_dir + 'c4-title.txt', 'r', encoding='utf-8') as f:
#     title_list = f.read().splitlines()
with open(data_dir + 'c4-abstract.json', 'r', encoding='utf-8') as f:
    abstract_list = json.load(f)
print('加载数据完成')

# 计算title
batch_size = 5000
n_p = 10
# docs = list(tqdm(nlp.pipe(title_list, batch_size=batch_size, disable=["ner", "parser", "textcat"], n_process=n_p), total=len(title_list),desc='计算title句子长度'))
# title_l_list = [len(doc) for doc in docs]

# title_sent_l_arr = np.array(title_l_list)
# np.save(data_dir + 'c4-title句子长度数组.npy', title_sent_l_arr)

# title_token_l_list = []

# for sent in tqdm(title_list, desc='计算title句子token长度'):
#     title_token_l_list.append(len(tokenizer.tokenize(sent)))

# title_token_l_arr = np.array(title_token_l_list)
# np.save(data_dir + 'c4-title句子token长度数组.npy', title_token_l_arr)

# 计算abstract

docs = list(tqdm(nlp.pipe(abstract_list, batch_size=batch_size, disable=["ner", "parser", "textcat"], n_process=n_p), total=len(abstract_list),desc='计算abstract句子长度'))
abstract_l_list = [len(doc) for doc in docs]

abstract_sent_l_arr = np.array(abstract_l_list)
np.save(data_dir + 'c4-abstract句子长度数组.npy', abstract_sent_l_arr)

abstract_token_l_list = []

for sent in tqdm(abstract_list, desc='计算title句子token长度'):
    abstract_token_l_list.append(len(tokenizer.tokenize(sent)))

abstract_token_l_arr = np.array(abstract_token_l_list)
np.save(data_dir + 'c4-abstract句子token长度数组.npy', abstract_token_l_arr)

In [ ]:
# 计算领域文本的长度
import os
import sys
from transformers import AutoTokenizer
from tqdm import tqdm
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
import numpy as np

output_file = "draw-data/c2-领域文本长度字典.npz"
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")
input_dir = '../Dataset-LyuCSE/wiki1m_{domain}.txt'
domain_list = ['chemistry','finance','medicine']
domain_length_dict = {}

for domain in domain_list:
    sent_l_arr = []
    input_file = input_dir.format(domain=domain)
    with open(input_file, 'r', encoding='utf-8') as f:
        sent_list = f.read().splitlines()

    for sent in tqdm(sent_list, desc='计算句子token长度'):
        sent_l_arr.append(len(tokenizer.tokenize(sent)))

    domain_length_dict[domain] = np.array(sent_l_arr)

np.savez(output_file, **domain_length_dict)
print(f"Saved to {output_file}")


In [ ]:
# C3-标签损失温度和权重画图
# 现在请按我给你的数据画图，需要画两个图，最终拼成一个图，左右排列，左边是温度，右边是权重，横坐标是对应的参数，即每个元组的第一个值，每个元组的第二个值时STS-B，第三个值是STS Avg.,所以对于每张图，需要两条折线，注意参数变化不是线性的，所以横坐标轴也不要是线性的，每个值都标出来，请你使用seaborn画图，使用绘画板自动配色，dpi600，直接给我注释好的代码，图上需要有legend，并且边框颜色是#2f3542，要求legend中没有title，要求横坐标和纵坐标都没有title，要求每张图的title在图片下方，现在数据如下，请你写代码：

temp_list = [(0.05, 77.39,79.2), (0.1, 77.2,79.5), (1, 76.8,79.05), (10, 76.68,78.6), (100, 78.31,77.91), (300, 76.1,77.8)]
weight_list = [(0.001, 76.19,77.18), (0.01, 76.46,78.11), (0.05, 76.99,78), (0.1, 77.17,79.09), (0.3, 78.31,77.91), (0.5, 76.78,79.1)]

In [17]:
# 计算均匀性和对齐性
import os
import sys
import numpy as np
from tqdm import tqdm
from datasets import load_dataset,Dataset
from scipy.spatial.distance import cdist
from transformers import AutoTokenizer,AutoModel
import torch
# 强制从项目目录出发
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from simcse.models import Pooler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "royokong/unsup-PromptBERT"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)

# prompt
eval_template = "This sentence of \"{sentence}\" means [MASK].".replace('[MASK]', tokenizer.mask_token)
pooler_type = "mask"

#eval_template = None
#pooler_type = "cls"
pooler = Pooler(pooler_type)

dataset_name = "mteb/stsbenchmark-sts"
dataset = load_dataset(dataset_name, split="test")

# 计算对齐性
# 过滤出score大于4分的
align_dataset = dataset.filter(lambda x: x['score'] > 4)

sent1 = align_dataset['sentence1']
sent2 = align_dataset['sentence2']
inputs = tokenizer(sent1+sent2, padding="longest", truncation=True, return_tensors="pt",max_length=512).to(device)
with torch.no_grad():
    outputs = model(**inputs)
last_hidden_state = outputs.last_hidden_state
pooler_output = last_hidden_state[:,0,:]
emb1 = pooler_output[:len(sent1)]
emb2 = pooler_output[len(sent1):]
# 归一化
emb1 = emb1 / emb1.norm(dim=-1, keepdim=True)
emb2 = emb2 / emb2.norm(dim=-1, keepdim=True)

# 计算对齐性
distances = torch.norm(emb1 - emb2, p=2, dim=1).cpu().numpy()
# 每项平方
distances = np.square(distances)
# 归一化
min = np.min(distances)
max = np.max(distances)
align = (distances - min) / (max - min)
align = align.mean()
# print(f"对齐性（align）：{align}")

# 计算均匀性
bs = 128
emb = None
for batch in tqdm(dataset.batch(bs)):
    sent1 = batch["sentence1"]
    sent2 = batch["sentence2"]

    if eval_template:
        sent1 = [eval_template.format(sentence=sent) for sent in sent1]
        sent2 = [eval_template.format(sentence=sent) for sent in sent2]

    inputs = tokenizer(sent1+sent2, padding="longest", truncation=True, return_tensors="pt",max_length=512).to(device)
    with torch.no_grad():
        outputs = model(**inputs)

    pooler_output = pooler(inputs['attention_mask'],outputs,inputs['input_ids'],mask_token_id=tokenizer.mask_token_id)

    # last_hidden_state = outputs.last_hidden_state
    # pooler_output = last_hidden_state[:,0,:]
    emb1 = pooler_output[:len(sent1)]
    emb2 = pooler_output[len(sent1):]
    emb = torch.cat([emb, emb1, emb2], dim=0) if emb is not None else torch.cat([emb1, emb2], dim=0)

# 计算两两欧氏距离
emb = emb.cpu().numpy()
emb = emb / np.linalg.norm(emb, axis=1, keepdims=True)  # 归一化
distances = cdist(emb, emb, metric='euclidean')
# 每项平方
distances = np.square(distances)
# 乘-2
distances = -2 * distances
# exp
distances = np.exp(distances)
uniform = np.log(distances.mean())

print(f"{model_name}")
print(f"对齐性（align）：{align}；均匀性（uniform）：{uniform}")


100%|██████████| 11/11 [00:01<00:00,  5.73it/s]


royokong/unsup-PromptBERT
对齐性（align）：0.28242960572242737；均匀性（uniform）：-1.5530707539998345


对齐性均匀性记录

princeton-nlp/unsup-simcse-bert-base-uncased: 对齐性（align）：0.23900701105594635；均匀性（uniform）：-2.613139836468747
princeton-nlp/unsup-simcse-bert-large-uncased：对齐性（align）：0.26787710189819336；均匀性（uniform）：-2.637618197241609
princeton-nlp/unsup-simcse-roberta-base：0.2593976557254791；均匀性（uniform）：-2.6227581643922027
princeton-nlp/unsup-simcse-roberta-large：对齐性（align）：0.24715536832809448；均匀性（uniform）：-3.171110722963605
sosuke/ease-bert-base-uncased：对齐性（align）：0.2508653402328491；均匀性（uniform）：-2.10836618622632
sosuke/ease-roberta-base-uncased：对齐性（align）：0.2293112576007843；均匀性（uniform）：-2.255434271018366
royokong/unsup-PromptBERT：对齐性（align）：0.28242960572242737；均匀性（uniform）：-1.5530707539998345
royokong/unsup-PromptRoBERTa：对齐性（align）：0.24216757714748383；均匀性（uniform）：-1.2456124612575867